In [2]:
import pandas as pd
import numpy as np
from prophet import Prophet
from tqdm import tqdm
from functools import reduce
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import VotingRegressor

In [3]:
df=pd.read_csv("../ISI_dataset\merged_wheat_reservoir.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\shrey\AppData\Local\Temp\ipykernel_30448\2216920933.py:1: SyntaxWarning: invalid escape sequence '\m'
  df=pd.read_csv("../ISI_dataset\merged_wheat_reservoir.csv")


,state_name,crop_name,apy_item_interval_start,temperature_recorded_date,state_temperature_max_val,state_temperature_min_val,state_rainfall_val,yield,FRL,Live Cap FRL,Level,Current Live Storage
0,Andhra Pradesh,wheat,2000,2000-01-01,30.38,14.47,0.0,0.57038,152.296667,2.838333,266.30,6.390
1,Andhra Pradesh,wheat,2000,2000-01-02,30.04,13.96,0.0,0.57038,152.296667,2.838333,266.18,6.330
2,Andhra Pradesh,wheat,2000,2000-01-03,29.92,12.98,0.0,0.57038,152.296667,2.838333,266.09,6.286
3,Andhra Pradesh,wheat,2000,2000-01-04,29.98,12.23,0.0,0.57038,152.296667,2.838333,266.03,6.257
4,Andhra Pradesh,wheat,2000,2000-01-05,29.77,13.24,0.0,0.57038,152.296667,2.838333,265.97,6.228


In [4]:
df['temperature_recorded_date'] = pd.to_datetime(df['temperature_recorded_date'])
df['year'] = df['temperature_recorded_date'].dt.year

In [5]:
# Use only data till 2022 for training
df = df[df['year'] < 2023].copy()

# Drop unreliable states
df = df[~df['state_name'].isin(['Odisha', 'Tamil Nadu'])]

# Group annually to match 2023 structure
df_annual = df.groupby(['state_name', 'crop_name', 'year']).agg({
    'state_rainfall_val': 'sum',
    'state_temperature_max_val': 'mean',
    'state_temperature_min_val': 'mean',
    'Live Cap FRL': 'mean',
    'FRL': 'mean',
    'Level': 'mean',
    'Current Live Storage': 'mean',
    'yield': 'mean'
}).reset_index()

In [6]:
df_annual.head()

,state_name,crop_name,year,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,yield
0,Andhra Pradesh,wheat,2000,942.03,34.738197,18.885082,2.838333,152.296667,179.065178,2.636298,0.57038
1,Andhra Pradesh,wheat,2001,918.85,35.137890,18.995397,2.838333,152.296667,171.736301,1.820863,0.69122
2,Andhra Pradesh,wheat,2002,654.57,35.292247,18.858493,2.838333,152.296667,169.014014,1.402067,0.92275
3,Andhra Pradesh,wheat,2003,832.53,35.550356,19.248548,2.838333,152.296667,161.787795,0.781289,0.29723
4,Andhra Pradesh,wheat,2004,786.89,34.954836,18.399536,2.838333,152.296667,164.511298,1.314224,0.22307


In [7]:
# One-hot encode 'state_name'
df_encoded = pd.get_dummies(df_annual, columns=['state_name'])

# Define features: original + one-hot encoded state columns
state_columns = [col for col in df_encoded.columns if col.startswith('state_name_')]

In [8]:
# Define features and target
features = ['state_rainfall_val', 'state_temperature_max_val', 'state_temperature_min_val', 'Live Cap FRL', 'FRL','Level','Current Live Storage']+ state_columns

# Split manually by year
train_df = df_encoded[df_encoded['year'] <= 2020]
test_df = df_encoded[df_encoded['year'].between(2021, 2022)]

In [9]:
X_train = train_df[features]
y_train = train_df['yield']
X_test = test_df[features]
y_test = test_df['yield']

In [10]:
# Models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR()
}

# Results container
results = []

# Loop through models
for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        'Model': name,
        'Train R²': round(train_r2, 4),
        'Test R²': round(test_r2, 4),
        'Train RMSE': round(train_rmse, 2),
        'Test RMSE': round(test_rmse, 2)
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Test R²', ascending=False))

                      Model  Train R²  Test R²  Train RMSE  Test RMSE
0         Linear Regression    0.8006   0.7091        0.42       0.50
2                   XGBoost    1.0000   0.6022        0.00       0.58
3         Gradient Boosting    0.9609   0.5936        0.18       0.59
1             Random Forest    0.9716   0.5678        0.16       0.61
4  Support Vector Regressor    0.1990  -0.1131        0.83       0.97


In [17]:
# Initialize individual models
lr = LinearRegression()

xgb = XGBRegressor(random_state=42)

# Ensemble model
ensemble = VotingRegressor(estimators=[
    ('lr', lr),
    ('xgb', xgb)
])

# Fit ensemble
ensemble.fit(X_train, y_train)

# Predict
train_pred = ensemble.predict(X_train)
test_pred = ensemble.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("📊 Ensemble Performance:")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")


📊 Ensemble Performance:
Train R²: 0.9500, Test R²: 0.6916
Train RMSE: 0.2082, Test RMSE: 0.5120


In [12]:
# --- 1. Define function to forecast any single feature using Prophet ---
def forecast_feature_prophet(df, feature_name):
    forecast_data = []

    for (state, crop), group in tqdm(df.groupby(['state_name', 'crop_name'])):
        yearly_data = group.groupby('year')[feature_name].mean().reset_index()

        if yearly_data.shape[0] < 4:
            continue

        prophet_df = yearly_data.rename(columns={'year': 'ds', feature_name: 'y'})
        prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

        try:
            model = Prophet()
            model.fit(prophet_df)

            future = pd.DataFrame({'ds': [pd.to_datetime('2023')]})
            forecast = model.predict(future)
            yhat = forecast['yhat'].values[0]

            forecast_data.append({
                'state_name': state,
                'crop_name': crop,
                feature_name: yhat
            })
        except:
            continue

    return pd.DataFrame(forecast_data)

# --- 2. Forecast each feature separately ---
df_rain = forecast_feature_prophet(df, 'state_rainfall_val')
df_temp_max = forecast_feature_prophet(df, 'state_temperature_max_val')
df_temp_min = forecast_feature_prophet(df, 'state_temperature_min_val')
df_livecap = forecast_feature_prophet(df, 'Live Cap FRL')
df_frl = forecast_feature_prophet(df, 'FRL')
df_level = forecast_feature_prophet(df, 'Level')
df_cls = forecast_feature_prophet(df, 'Current Live Storage')

# --- 3. Merge all forecasted dataframes ---
from functools import reduce
dfs = [df_rain, df_temp_max, df_temp_min, df_livecap, df_frl, df_level, df_cls]
df_2023 = reduce(lambda left, right: pd.merge(left, right, on=['state_name', 'crop_name'], how='outer'), dfs)

# --- 4. One-hot encode state_name ---
df_2023_encoded = df_2023.copy()  # Keep original columns
state_names = df_2023_encoded[['state_name', 'crop_name']]  # Keep for merging later

df_2023_encoded = pd.get_dummies(df_2023_encoded, columns=['state_name'])
df_2023_encoded = pd.concat([state_names, df_2023_encoded.drop(columns=['crop_name'])], axis=1)


  0%|          | 0/12 [00:00<?, ?it/s]22:20:15 - cmdstanpy - INFO - Chain [1] start processing
22:20:15 - cmdstanpy - INFO - Chain [1] done processing
  8%|▊         | 1/12 [00:00<00:06,  1.76it/s]22:20:15 - cmdstanpy - INFO - Chain [1] start processing
22:20:15 - cmdstanpy - INFO - Chain [1] done processing
 17%|█▋        | 2/12 [00:00<00:03,  2.72it/s]22:20:16 - cmdstanpy - INFO - Chain [1] start processing
22:20:16 - cmdstanpy - INFO - Chain [1] done processing
 25%|██▌       | 3/12 [00:01<00:03,  2.71it/s]22:20:16 - cmdstanpy - INFO - Chain [1] start processing
22:20:16 - cmdstanpy - INFO - Chain [1] done processing
 33%|███▎      | 4/12 [00:01<00:02,  3.15it/s]22:20:16 - cmdstanpy - INFO - Chain [1] start processing
22:20:16 - cmdstanpy - INFO - Chain [1] done processing
 42%|████▏     | 5/12 [00:01<00:02,  3.23it/s]22:20:16 - cmdstanpy - INFO - Chain [1] start processing
22:20:17 - cmdstanpy - INFO - Chain [1] done processing
 50%|█████     | 6/12 [00:01<00:01,  3.54it/s]22:20:17

In [13]:
df_2023_encoded.head()

,state_name,crop_name,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,state_name_Andhra Pradesh,...,state_name_Gujarat,state_name_Jharkhand,state_name_Karnataka,state_name_Madhya Pradesh,state_name_Maharashtra,state_name_Rajasthan,state_name_Telangana,state_name_Uttar Pradesh,state_name_Uttarakhand,state_name_West Bengal
0,Andhra Pradesh,wheat,2.774297,34.250960,19.317292,2.396822,183.580095,140.482266,1.072416,True,...,False,False,False,False,False,False,False,False,False,False
1,Chhattisgarh,wheat,3.704114,33.805702,18.200942,1.365667,377.820000,353.377518,1.355013,False,...,False,False,False,False,False,False,False,False,False,False
2,Gujarat,wheat,2.706400,34.828181,18.653730,0.811491,115.199781,119.286907,0.366662,False,...,True,False,False,False,False,False,False,False,False,False
3,Jharkhand,wheat,3.394479,33.182895,18.405240,0.404750,296.267500,290.707379,0.200885,False,...,False,True,False,False,False,False,False,False,False,False
4,Karnataka,wheat,3.444745,33.812358,17.518092,1.539062,597.731875,586.334229,0.790733,False,...,False,False,True,False,False,False,False,False,False,False


In [14]:
X_2023 = df_2023_encoded[features]

# Use your trained ensemble model
y_2023_pred = ensemble.predict(X_2023)

# Add prediction to the dataframe
df_2023_encoded['predicted_yield'] = y_2023_pred

# Select output
output_2023 = df_2023_encoded[['crop_name'] + [col for col in df_2023_encoded.columns if col.startswith('state_name_')] + ['predicted_yield']]


In [15]:
# Convert dummy columns back to state_name
state_names = df_2023_encoded[[col for col in df_2023_encoded.columns if col.startswith('state_name_')]].idxmax(axis=1)
state_names = state_names.str.replace('state_name_', '')

# Final output
final_2023_yield = pd.DataFrame({
    'state_name': state_names,
    'crop_name': df_2023_encoded['crop_name'],
    'predicted_yield_2023': df_2023_encoded['predicted_yield']
})

print(final_2023_yield)
final_2023_yield.to_csv("../yield_prediction.csv", index=False)

        state_name crop_name  predicted_yield_2023
0   Andhra Pradesh     wheat              1.412746
1     Chhattisgarh     wheat              1.354791
2          Gujarat     wheat              2.784415
3        Jharkhand     wheat              1.816802
4        Karnataka     wheat              1.274825
5   Madhya Pradesh     wheat              2.571427
6      Maharashtra     wheat              1.461962
7        Rajasthan     wheat              2.906355
8        Telangana     wheat              1.608218
9    Uttar Pradesh     wheat              2.731088
10     Uttarakhand     wheat              2.131276
11     West Bengal     wheat              2.489650


In [16]:
# Step 1: Get actual yields from 2019 to 2022
df_recent = df_annual[df_annual['year'].between(2019, 2022)].copy()

# Pivot to get each year's yield as a column
yield_table = df_recent.pivot_table(
    index=['state_name', 'crop_name'],
    columns='year',
    values='yield'
).reset_index()

# Rename columns for clarity
yield_table = yield_table.rename(columns={
    2019: 'yield_2019',
    2020: 'yield_2020',
    2021: 'yield_2021',
    2022: 'yield_2022'
})

# Step 2: Prepare 2023 predicted yield
df_2023_yield = df_2023_encoded[['state_name', 'crop_name', 'predicted_yield']].copy()
df_2023_yield = df_2023_yield.rename(columns={'predicted_yield': 'yield_2023'})

# Step 3: Merge the 2023 predicted yield into the table
final_yield_table = pd.merge(yield_table, df_2023_yield, on=['state_name', 'crop_name'], how='left')

# Display final table
print(final_yield_table)


        state_name crop_name  yield_2019  yield_2020  yield_2021  yield_2022  \
0   Andhra Pradesh     wheat     0.29730     1.53333     1.15152     1.22034   
1     Chhattisgarh     wheat     1.16869     1.54934     1.27183     1.65277   
2          Gujarat     wheat     3.26783     3.20477     3.20500     3.17436   
3        Jharkhand     wheat     2.04582     2.33701     2.28233     2.14552   
4        Karnataka     wheat     1.13767     1.29158     1.28652     1.37338   
5   Madhya Pradesh     wheat     3.67123     3.62850     3.46815     3.57601   
6      Maharashtra     wheat     1.69678     1.83930     1.89380     1.91175   
7        Rajasthan     wheat     3.97297     3.91534     3.91265     3.76197   
8        Telangana     wheat     1.84232     2.63998     2.00913     2.07280   
9    Uttar Pradesh     wheat     3.67517     3.80439     3.73385     3.73492   
10     Uttarakhand     wheat     2.92611     3.15289     3.02250     2.91634   
11     West Bengal     wheat     2.70817